In [ ]:
import glob
import re
import fractions
from typing import Any
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
PATH = r'D:\Research\Learning_CFT\ADE-classification\chtc_evaluation\cft_lib_3M_c50_rand\epochs'
EPOCH_RANGE = 99

In [ ]:
def get_bracket_block(line: str) -> str | None:
    if 'None' in line:
        return None
    m = re.search(r'\[\[.*\]\]', line)
    return m.group(0) if m else None


def parse_block(expr: str | None) -> Any:
    if expr is None:
        return None
    try:
        return eval(expr.replace('Fraction', 'fractions.Fraction'), {'fractions': fractions})
    except Exception:
        return None


def make_canonical(obj: Any) -> Any:
    """Recursively convert lists to sorted tuples so element order is irrelevant."""
    if isinstance(obj, list):
        return tuple(sorted([make_canonical(e) for e in obj], key=repr))
    if isinstance(obj, tuple):
        return tuple(make_canonical(e) for e in obj)
    return obj


# --- per-line extraction functions ---

def _valid_items(obj):
    """Filter out malformed elements (e.g. bare ints) that the model occasionally emits."""
    return [item for item in obj if isinstance(item, (list, tuple))] if obj else None


def extract_labels(line: str) -> list | None:
    obj = _valid_items(parse_block(get_bracket_block(line)))
    return sorted([item[0] for item in obj]) if obj else None


def extract_fractions(line: str) -> list | None:
    obj = _valid_items(parse_block(get_bracket_block(line)))
    return sorted([item[1] for item in obj]) if obj else None


def extract_full(line: str) -> str | None:
    raw = get_bracket_block(line)
    return re.sub(r'\s+', '', raw) if raw else None


def extract_order_free(line: str) -> Any:
    obj = parse_block(get_bracket_block(line))
    return make_canonical(obj) if obj else None


def evaluate_files(file_list: list[str], extract_fn) -> list[float | None]:
    accuracies = []
    for file in file_list:
        correct, total = 0, 0
        with open(file, 'r', encoding='utf-8') as f:
            lines = f.read().splitlines()
        tgt = pred = None
        for line in lines:
            if line.startswith('tgt'):
                tgt = extract_fn(line)
            elif line.startswith(('0', '1')):
                pred = extract_fn(line)
                if tgt is not None and pred is not None:
                    correct += (tgt == pred)
                    total += 1
        accuracies.append(correct / total if total else None)
    return accuracies

In [ ]:
file_list = sorted(
    glob.glob(f'{PATH}/eval.valid.cfts.*'),
    key=lambda x: int(x.split('.')[-1])
)

label_acc      = evaluate_files(file_list, extract_labels)
frac_acc       = evaluate_files(file_list, extract_fractions)
full_acc       = evaluate_files(file_list, extract_full)
order_free_acc = evaluate_files(file_list, extract_order_free)

In [ ]:
epochs = [-1] + list(range(EPOCH_RANGE))

def to_pct(acc_list):
    return [0] + [x * 100 for x in acc_list[:EPOCH_RANGE]]

label_pct      = to_pct(label_acc)
frac_pct       = to_pct(frac_acc)
full_pct       = to_pct(full_acc)
order_free_pct = to_pct(order_free_acc)

In [ ]:
plt.figure(figsize=(7, 6))
plt.plot(epochs, full_pct,       linestyle='-', color='blue',  label='model accuracy',                linewidth=1)
plt.plot(epochs, order_free_pct, linestyle='-', color='green', label='permutation-invariant accuracy', linewidth=1)
plt.xlabel('Epoch', fontsize=18)
plt.ylabel('Accuracy (%)', fontsize=18)
plt.xticks(fontsize=17)
plt.yticks(fontsize=17)
plt.legend(loc='lower right', bbox_to_anchor=(1, 0.08), fontsize=16)
plt.show()

## Comparison: fixed training vs. random training

Run the cells above with a different `PATH` to populate a second set of results,
then store them below before re-running.

In [ ]:
# Store current run's results before switching PATH
fixed_full_pct       = full_pct
fixed_order_free_pct = order_free_pct

In [ ]:
plt.figure(figsize=(7, 6))
plt.plot(epochs, fixed_full_pct, linestyle='-', color='blue',  label='fixed training',  linewidth=1)
plt.plot(epochs, full_pct,       linestyle='-', color='green', label='random training', linewidth=1)
plt.xlabel('Epoch', fontsize=18)
plt.ylabel('Accuracy (%)', fontsize=18)
plt.xticks(fontsize=17)
plt.yticks(fontsize=17)
plt.legend(loc='lower right', bbox_to_anchor=(1, 0.08), fontsize=16)
plt.savefig('cft_KM_c50_fixedVSrandom_full.pdf')
plt.show()